# Bonus — выбросы и прогноз заказов · 10 баллов

Статус: проектирование и первичное исследование схемы данных. Источник — PDF, страницы 4–5.
В шапке требуется 3 notebooks, а третья задача названа bonus. Подготовлен отдельный notebook для бонуса; обязательность его сдачи остаётся неоднозначной.

# 1. Data Reading, Exploration and Preprocessing

Локальный ZIP содержит SQLite F26.ML.Assignment.One.data.db размером 206979072 байта.
Проверено read-only: order_history — 517610 строк, 517610 distinct order_id, 1419 distinct store_id. Строковые min/max DATE_CREATE: 2024-05-08 00:01:12 и 2024-09-05 15:25:25. Требуется дополнительно проверить парсинг дат и полноту временного покрытия.
Таблицы: order_history, order_busket (именно такое написание), products, order_props_value, order_props.
region_id найден в order_props с ID=11. Соединять order_history.order_id с order_props_value.order_id при ORDER_PROPS_ID=11, VALUE — идентификатор региона. Перед join проверить кратность property, пропуски и конфликты, чтобы не размножить заказы.
Цель — число новых distinct order_id по DATE_CREATE на регион и временной интервал. Статусы, состав корзины и будущие timestamps не считать отдельными заказами. Проверить часовой пояс и смысл DATE_CREATE по документации из лабораторных.
Регулярная временная сетка, анализ нулей и пропусков, суточной/недельной сезонности. Нулевой count допустим при подтверждённой полноте данных.

# 2. Outlier Detection · 4 балла

Предлагаемые два метода: causal rolling median/MAD (Hampel) и пороги по остаткам от сезонного baseline, оценённого на прошлых данных.
Сравнить долю выбросов, совпадение методов и графики. Высокий спрос может быть настоящим событием: обосновать каждый способ обработки.
Предлагаем clipping либо imputation по прошлой сезонной медиане с сохранением регулярной сетки. Параметры детекторов обучать отдельно внутри каждого train fold. Не использовать будущие точки rolling window.
Для честной оценки сохранять исходные validation targets. Обработку применять к train и доступной на момент прогноза истории, отдельно раскрывать подтверждённые ошибки измерений.

# 2.1. Model Definition, Training and Hyperparameters · 6 баллов вместе с оценкой

Предлагаемые горизонты n=15 и n=60 минут. Target для момента t — сумма заказов региона в (t,t+n]. Это решение нужно проверить с документацией данных.
Baseline — seasonal naive с суточным либо недельным периодом, выбранным на train/validation. DL — PyTorch GRU: история count и календарных признаков → GRU → Linear → неотрицательный прогноз суммы. Один общий GRU по регионам с region embedding либо отдельные модели, выбор обосновать объёмом данных.
Входы содержат только признаки, доступные к t. Scaler fit на train. Предлагаемый lookback выбирается по rolling validation. Логировать losses и validation metrics в TensorBoard.

# 3. Model Performance Evaluation

Rolling-origin validation с возрастающим train и последующим validation. Ни одно train-окно с target (t,t+n] не должно пересекать validation boundary; предусмотреть зазор по горизонту.
Сравнить два горизонта × baseline/GRU × raw/метод1/метод2 обработки, минимум 12 комбинаций до учёта folds.
Таблица MAE/RMSE по каждому региону и агрегированно, средние по folds и разброс. Показать наблюдения и прогнозы, влияние выбросов и обработок. Если есть final holdout, сохранить последний временной отрезок до завершения выбора настроек.

# 4. Conclusion and Possible Improvements

Обосновать методы outlier handling, влияние на каждый горизонт и модель. Указать ограничения идентификации региона, временного покрытия и семантики timestamps.
Перед сдачей выполнить все клетки и сохранить графики, таблицы, выводы и TensorBoard. Текущий файл содержит план реализации, результаты экспериментов появятся после выполнения кода.